In [3]:
import cv2
import numpy as np
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.optim as optim
import os
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import csv

Mỗi ảnh được xử lý theo quy trình sau:
-  Chuyển sang grayscale do nhiều đặc trưng bệnh lý quan trọng trong OCT thể hiện ở biến thiên cường độ và cấu trúc lớp võng mạc; OCT thường là ảnh xám.
-  Resize về 224×224 chuẩn phổ biến cho CNN.
-  Bilateral Filter (d=9, sigmaColor=75, sigmaSpace=75) giảm nhiễu trong khi vẫn giữ biên. Trong ảnh OCT võng mạc, bảo tồn biên rất quan trọng vì các cấu trúc nhỏ (vùng CNV) có thể là biên rõ ràng. Kích thước kernel là 9 là vừa đủ để giảm nhiễu mà vẫn ít làm mờ.
-  CLAHE (clipLimit=2.0, tileGridSize=(8,8)) (Contrast Limited Adaptive Histogram Equalization) giúp tăng cường tương phản cục bộ, giúp vùng có chi tiết nhỏ (có độ tương phản thấp) nổi bật. tileGridSize: kích thước kernel (8,8) tốt cho ảnh 224×224.
-  Laplacian Sharpening (lap = cv2.Laplacian; img = 0.8img + 0.2|lap|): làm nổi biên, giúp CNN dễ học các đặc trưng cạnh, cấu trúc lớp võng mạc. Tỷ lệ 8/2 giúp làm nổi bật phần biên vừa phải mà không làm ảnh quá nhiễu.
-  Chuẩn hóa / scale to [0,1] giúp gradient ổn định, train tốt hơn.


In [4]:
def img_proccessing(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

    
    img = cv2.resize(img, dsize=(224,224), interpolation=cv2.INTER_AREA)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    lap = cv2.Laplacian(img, cv2.CV_64F)
    img = cv2.addWeighted(img, 0.8, cv2.convertScaleAbs(lap), 0.2, 0)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    return img


Bước tiền xử lý ảnh và chuẩn bị dữ liệu

In [5]:
class OCTDataset(Dataset):
    def __init__(self, data_dir, categories):
        self.data_dir = data_dir
        self.categories = categories
        self.image_paths = []
        self.labels = []

        for label, category in enumerate(categories):
            folder = os.path.join(data_dir, category)
            for fname in os.listdir(folder):
                self.image_paths.append(os.path.join(folder, fname))
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # ---- xử lý ảnh ----
        img = img_proccessing(img_path)
        img = torch.from_numpy(img).permute(2, 0, 1)  

        return img.float(), torch.tensor(label, dtype=torch.float32)


Chia tập dữ liệu

In [ ]:
DATA_DIR = "Dataset"
categories = ["NORMAL", "CNV"]

dataset = OCTDataset(DATA_DIR, categories)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

NUM_WORKERS = 0  
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

NameError: name 'train_dataset' is not defined

Mô hình CNN

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),

            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(self.conv(x))


CNN mỏng hơn

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super(ShallowCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 64),

            nn.ReLU(),
            nn.Linear(64, 1),
            
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(self.conv(x))


CNN có Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
        )
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        return torch.relu(self.conv(x) + self.shortcut(x))


class ResidualCNN(nn.Module):
    def __init__(self):
        super(ResidualCNN, self).__init__()
        self.layer1 = ResidualBlock(1, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.layer2 = ResidualBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.layer3 = ResidualBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),

            nn.ReLU(),
            nn.Linear(128, 1),
            
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))
        return self.fc(x)


Lưu mô hình

In [8]:
def save_models_and_history(models, history, weight_dir="weights", log_file="models_history.csv", summary_file="models_summary.csv"):
    import os
    os.makedirs(weight_dir, exist_ok=True)


    for name, model in models.items():
        torch.save(model.state_dict(), os.path.join(weight_dir, f"{name}_weights.pth"))
        print(f" Saved weights for {name} -> {weight_dir}/{name}_weights.pth")

   
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["model", "epoch", "train_loss", "val_loss", "val_acc_pct", "val_auc"])
        for name, stats in history.items():
            for i in range(len(stats["train_loss"])):
                writer.writerow([
                    name,
                    i + 1,
                    stats["train_loss"][i],
                    stats["val_loss"][i],
                    stats["val_acc"][i],
                    stats["val_auc"][i]
                ])
    print(f"Saved detailed training history -> {log_file}")

  
    with open(summary_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["model", "total_time_s", "avg_time_per_epoch_s"])
        for name, stats in history.items():
            writer.writerow([name, stats["total_time"], stats["avg_time"]])
    print(f"Saved training time summary -> {summary_file}")

Huấn luyện mô hình

In [ ]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
EPOCHS = 10

models = {
    "SimpleCNN": SimpleCNN(),
    "ShallowCNN": ShallowCNN(),
    "ResidualCNN": ResidualCNN()
}

history = {}

for name, model in models.items():
    print(f"Training {name}...")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_losses, val_losses, val_accs, val_aucs = [], [], [], []
    
    start_time = time.time()

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        # --- training ---
        model.train()
        total_loss, total_samples = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            total_samples += imgs.size(0)
        train_loss = total_loss / total_samples

        # --- validation ---
        model.eval()
        val_total_loss, val_total, correct = 0, 0, 0
        all_labels, all_probs = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_total_loss += loss.item() * imgs.size(0)
                val_total += imgs.size(0)
                preds = (outputs > 0.5).float()
                correct += (preds == labels).sum().item()
                all_labels.extend(labels.cpu().numpy().flatten().tolist())
                all_probs.extend(outputs.cpu().numpy().flatten().tolist())

        val_loss = val_total_loss / val_total
        val_acc = correct / val_total
        try:
            if len(set(all_labels)) > 1:
                fpr, tpr, _ = roc_curve(np.array(all_labels), np.array(all_probs))
                val_auc = auc(fpr, tpr)
            else:
                val_auc = float('nan')
        except:
            val_auc = float('nan')

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc * 100)
        val_aucs.append(val_auc)

        epoch_time = time.time() - epoch_start
        print(f"{name} Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
              f"val_acc={val_acc:.2f}, val_auc={val_auc:.4f}, epoch_time={epoch_time:.2f}s")

    total_time = time.time() - start_time
    avg_time = total_time / EPOCHS
    print(f" {name} finished training in {total_time:.2f}s "
          f"(~{avg_time:.2f}s per epoch)\n")

    history[name] = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "val_acc": val_accs,
        "val_auc": val_aucs,
        "total_time": total_time,
        "avg_time": avg_time
    }
    save_models_and_history({name: model}, {name: history[name]})

Using device: cuda
Training SimpleCNN...
SimpleCNN Epoch 1: train_loss=0.2293, val_loss=0.1042, val_acc=0.96, val_auc=0.9936, epoch_time=1872.46s
SimpleCNN Epoch 2: train_loss=0.0876, val_loss=0.0620, val_acc=0.98, val_auc=0.9970, epoch_time=2008.63s
SimpleCNN Epoch 3: train_loss=0.0549, val_loss=0.0469, val_acc=0.99, val_auc=0.9981, epoch_time=735.30s
SimpleCNN Epoch 4: train_loss=0.0376, val_loss=0.0378, val_acc=0.99, val_auc=0.9987, epoch_time=1777.83s
SimpleCNN Epoch 5: train_loss=0.0268, val_loss=0.0339, val_acc=0.99, val_auc=0.9988, epoch_time=1713.32s
SimpleCNN Epoch 6: train_loss=0.0201, val_loss=0.0489, val_acc=0.99, val_auc=0.9987, epoch_time=1583.54s
SimpleCNN Epoch 7: train_loss=0.0143, val_loss=0.0416, val_acc=0.99, val_auc=0.9992, epoch_time=1838.97s
SimpleCNN Epoch 8: train_loss=0.0134, val_loss=0.0465, val_acc=0.99, val_auc=0.9990, epoch_time=1259.85s
SimpleCNN Epoch 9: train_loss=0.0064, val_loss=0.0480, val_acc=0.99, val_auc=0.9991, epoch_time=2008.75s
SimpleCNN Epoch

NameError: name 'save_models_and_history' is not defined